# Silver Layer: Matchday Stats by League

This pipeline reads from the bronze table `workspace.football_data_project.bronze_matchday_stats` and splits the data into five separate silver tables — one per league — for the top five European football leagues. Each silver table applies data quality transformations such as null handling, derived columns, and standardised naming.

In [0]:
# ── Silver Layer Configuration ──────────────────────────────────────────────
# Top 5 European leagues with their API-Football league IDs

LEAGUES = {
    39:  "premier_league",     # England  – Premier League
    140: "la_liga",            # Spain    – La Liga
    78:  "bundesliga",         # Germany  – Bundesliga
    135: "serie_a",            # Italy    – Serie A
    61:  "ligue_1",            # France   – Ligue 1
}

BRONZE_TABLE = "workspace.football_data_project.bronze_matchday_stats"
SILVER_SCHEMA = "workspace.football_data_project"

print(f"Source:  {BRONZE_TABLE}")
print(f"Target:  {SILVER_SCHEMA}.silver_<league>_matchday_stats")
print(f"Leagues: {len(LEAGUES)}")

In [0]:
# ── Create silver tables (one per league) ────────────────────────────────────
# Each silver table:
#   • filters the bronze source by league_id
#   • adds league_name and silver_processing_time columns
#   • casts nullable numeric stats to 0 where null (player didn't record them)
#   • trims string columns

from pyspark.sql.functions import col, lit, when, current_timestamp, trim

bronze_df = spark.table(BRONZE_TABLE)

# Columns that should default to 0 instead of null
numeric_stat_cols = [
    "offsides", "shots_total", "shots_on", "goals_total", "goals_conceded",
    "goals_assists", "goals_saves", "passes_total", "passes_key",
    "tackles_total", "tackles_blocks", "tackles_interceptions",
    "duels_total", "duels_won", "dribbles_attempts", "dribbles_success",
    "dribbles_past", "fouls_drawn", "fouls_committed",
    "penalty_won", "penalty_commited", "penalty_missed", "penalty_saved",
]

for league_id, league_slug in LEAGUES.items():
    silver_df = (
        bronze_df
        .filter(col("league_id") == league_id)
        .withColumn("league_name", lit(league_slug.replace("_", " ").title()))
        # Fill null numeric stats with 0
        .fillna(0, subset=numeric_stat_cols)
        # Trim whitespace on string columns
        .withColumn("round", trim(col("round")))
        .withColumn("team_name", trim(col("team_name")))
        .withColumn("player_name", trim(col("player_name")))
        .withColumn("games_position", trim(col("games_position")))
        # Add processing timestamp
        .withColumn("silver_processing_time", current_timestamp())
    )

    table_name = f"{SILVER_SCHEMA}.silver_{league_slug}_matchday_stats"
    silver_df.write.mode("overwrite").saveAsTable(table_name)
    row_count = spark.table(table_name).count()
    print(f"  ✓ {table_name:<65} {row_count:>6,} rows")

print("\nSilver layer pipeline complete.")

In [0]:
# ── Verify silver tables ──────────────────────────────────────────────────────

print("Silver table summary:")
print("=" * 80)

for league_id, league_slug in LEAGUES.items():
    table_name = f"{SILVER_SCHEMA}.silver_{league_slug}_matchday_stats"
    try:
        df = spark.table(table_name)
        count = df.count()
        distinct_matchdays = df.select("round").distinct().count()
        distinct_fixtures = df.select("fixture_id").distinct().count()
        distinct_teams = df.select("team_id").distinct().count()
        print(f"  {table_name}")
        print(f"    rows={count:>6,}  matchdays={distinct_matchdays:>3}  fixtures={distinct_fixtures:>3}  teams={distinct_teams:>3}")
    except Exception as e:
        print(f"  {table_name} → ERROR: {e}")

print("=" * 80)